<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/%EA%B0%95%EC%9D%98_8%EA%B8%B0_AI%EA%B0%9C%EB%A1%A0_6%EC%B0%A8%EC%8B%9C_02_%ED%9A%8C%EA%B7%80%EC%86%90%EC%8B%A4%ED%95%A8%EC%88%98_%EB%B9%84%EA%B5%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

회귀 손실 함수

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [27]:
# 재현성 위한 시드 고정
torch.manual_seed(42)
np.random.seed(42)

In [28]:
# 간단한 예측값과 실제값
y_true = torch.tensor([10.0, 20.0, 30.0])
y_pred = torch.tensor([12.0, 19.0, 35.0])

In [29]:
# mse 계산
mse_loss = nn.MSELoss()
mse_loss(y_pred, y_true)
mse_value = mse_loss(y_pred, y_true)
print(mse_value)

tensor(10.)


In [30]:
# rmse 계산
mse_loss = nn.MSELoss()
mse_loss(y_pred, y_true)
mse_value = mse_loss(y_pred, y_true)
print(mse_value)
print(np.sqrt(mse_value.numpy()))   # rmse

tensor(10.)
3.1622777


In [31]:
# mae 계산
mae_loss = nn.L1Loss()
mae_loss(y_pred, y_true)
mae_value = mae_loss(y_pred, y_true)
print(mae_value)

tensor(2.6667)


In [32]:
# huber loss 계산
huber_loss = nn.HuberLoss(delta=1.0)
huber_value = huber_loss(y_pred, y_true)
print(huber_value)

tensor(2.1667)


In [33]:
print(y_pred.numpy())   # 예측값
print(y_true.numpy())   # 실제값
print((y_pred - y_true).numpy())    # 오차(error)

[12. 19. 35.]
[10. 20. 30.]
[ 2. -1.  5.]


In [34]:
print(mse_value.numpy())
print(mae_value.numpy())
print(huber_value.numpy())

10.0
2.6666667
2.1666667


In [35]:
# 수동 계산으로 검증
(y_pred - y_true).numpy()   # loss

array([ 2., -1.,  5.], dtype=float32)

In [36]:
errors = (y_pred - y_true).numpy()
print(np.mean(errors ** 2)) # mse
print(np.mean(np.abs(errors)))  # mae

10.0
2.6666667


이상치가 있을 때 손실함수 비교

In [37]:
# 정상 데이터 + 이상치
y_true_with_outlier = torch.tensor([10.0, 20.0, 30.0, 40.0, 50.0])
y_pred_normal = torch.tensor([11.0, 19.0, 31.0, 39.0, 51.0])  # 정상 예측
y_pred_with_outlier = torch.tensor([11.0, 19.0, 31.0, 100.0, 51.0])  # 이상치 포함

In [38]:
# 정상 예측
print('실제 :',y_true_with_outlier.numpy())
print('예측 :',y_pred_with_outlier.numpy())

실제 : [10. 20. 30. 40. 50.]
예측 : [ 11.  19.  31. 100.  51.]


In [44]:
# 각 회귀함수손실 함수 적용
mse_normal = mse_loss(y_pred_normal, y_true_with_outlier)
mae_normal = mae_loss(y_pred_normal, y_true_with_outlier)
huber_normal = huber_loss(y_pred_normal, y_true_with_outlier)

print(mse_normal)
print(mae_normal)
print(huber_normal)

tensor(1.)
tensor(1.)
tensor(0.5000)


In [40]:
# 이상치 포함 예측
print('실제 :',y_true_with_outlier.numpy())
print('예측 :',y_pred_with_outlier.numpy())
print('오차 :',( y_true_with_outlier - y_pred_with_outlier).numpy())

실제 : [10. 20. 30. 40. 50.]
예측 : [ 11.  19.  31. 100.  51.]
오차 : [ -1.   1.  -1. -60.  -1.]


In [41]:
mse_outlier = mse_loss(y_pred_with_outlier, y_true_with_outlier)
mae_outlier = mae_loss(y_pred_with_outlier, y_true_with_outlier)
huber_outlier = huber_loss(y_pred_with_outlier, y_true_with_outlier)

In [42]:
print(mse_outlier)
print(mae_outlier)
print(huber_outlier)

tensor(720.8000)
tensor(12.8000)
tensor(12.3000)


In [47]:
# 분석 시 활용법
# 증가율 비교
print(f'증가율:{mse_outlier / mse_normal:.2f}배 증가')
print(f'증가율:{mae_outlier / mae_normal:.2f}배 증가')
print(f'증가율:{huber_outlier / huber_normal:.2f}배 증가')

증가율:720.80배 증가
증가율:12.80배 증가
증가율:24.60배 증가


In [ ]:
# 분석 결과
# mse는 이상치에 매운 민감한 것울 발겨 (제곱때문)
# mae는 이상치에 강건 (절대값, 즉 크기만 고려)
# huber중간 (작은 오차는 제곱, 큰 오차는 선형)

손실함수별 학습 비교

In [48]:
# 데이터 생성
X, y = make_regression(n_samples=500, n_features=10, noise=10.0, random_state=42)

In [50]:
# 이상치 추가 (10% 데이터에 큰 노이즈)
n_outlier = int(0.1 * len(y))
outlier_indices = np.random.choice(len(y), n_outlier, replace=False)
y[outlier_indices] += np.random.randn(n_outlier)*50 # 큰 노이즈

In [52]:
# 데이터 분할
X_train, X_test, y_train, y_test = \
train_test_split(X,y,test_size=0.2,random_state=42)

In [55]:
# 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
# print(X_train)
X_test = scaler.transform(X_test)

In [65]:
# 텐서변환
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)

X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

print(X_train_t.shape)
print(y_train_t.shape)
# print(y_train_f)

torch.Size([400, 10])
torch.Size([400, 1])


In [61]:
# 훈련 데이터
print(X_train.shape)

# 이상치
print(n_outlier)

(400, 10)
50


In [62]:
# 간단한 회귀 모델
class RegressionModel(nn.Module):
    """간단한 회귀 신경망"""
    def __init__(self):
        super(RegressionModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )


    def forward(self, x):
        return self.network(x)

In [63]:
# 세 가지 손실함수로 각각 학습
loss_functions = {
    'MSE': nn.MSELoss(),
    'MAE': nn.L1Loss(),
    'Huber': nn.HuberLoss(delta=1.0)
}

In [67]:
results = {}

for loss_name, criterion in loss_functions.items():
    print(f"\n{loss_name}로 학습 중...")


    # 모델 초기화
    model = RegressionModel()
    optimizer = optim.Adam(model.parameters(), lr=0.01)


    # 학습
    train_losses = []
    num_epochs = 100


    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()


        output = model(X_train_t)
        loss = criterion(output, y_train_t)


        loss.backward()
        optimizer.step()


        train_losses.append(loss.item())


    # 테스트
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)


        # 모든 지표로 평가
        test_mse = nn.MSELoss()(test_pred, y_test_t).item()
        test_mae = nn.L1Loss()(test_pred, y_test_t).item()
        test_huber = nn.HuberLoss()(test_pred, y_test_t).item()


    results[loss_name] = {
        'train_losses': train_losses,
        'test_mse': test_mse,
        'test_mae': test_mae,
        'test_huber': test_huber,
        'predictions': test_pred
    }


    print(f"  최종 훈련 손실: {train_losses[-1]:.4f}")
    print(f"  테스트 MSE: {test_mse:.4f}")
    print(f"  테스트 MAE: {test_mae:.4f}")



MSE로 학습 중...
  최종 훈련 손실: 151.4789
  테스트 MSE: 1333.2311
  테스트 MAE: 25.8018

MAE로 학습 중...
  최종 훈련 손실: 6.6869
  테스트 MSE: 1386.9045
  테스트 MAE: 26.4775

Huber로 학습 중...
  최종 훈련 손실: 6.3157
  테스트 MSE: 1292.3202
  테스트 MAE: 25.5389
